In [1]:
from text2text_coref.convert import read_data, reduce_discontinuous_mention
import json
from tqdm import tqdm

In [2]:
def extract_dataset(docs):
    dataset_docs = []

    for doc in docs:
        sentences_in_doc = []
        for bundle in doc.bundles:
            tree = bundle.get_tree()
            real_words = tree.descendants
            if not real_words:
                continue

            # Map real words to their 0-based positions
            ord_to_idx = {word.ord: i for i, word in enumerate(real_words)}
            sentence_tokens = [w.form for w in real_words]

            # 1. Map empty nodes to their SYNTACTIC HEAD with a LINEAR FALLBACK
            empty_to_real_map = {}
            for empty_node in tree.descendants_and_empty:
                if empty_node.is_empty():
                    # Attempt 1: Climb the dependency tree for a real word
                    anchor_node = empty_node.parent
                    while anchor_node and anchor_node.is_empty():
                        anchor_node = anchor_node.parent

                    # Attempt 2: If syntax fails (Root), find the nearest preceding real word
                    if anchor_node is None or anchor_node.is_root():
                        prev_node = None
                        for r_node in real_words:
                            if r_node.ord < empty_node.ord:
                                prev_node = r_node
                            else:
                                break

                        if prev_node:
                            empty_to_real_map[empty_node.ord] = ord_to_idx[prev_node.ord]
                        else:
                            # Absolute fallback if at the very beginning of the sentence
                            empty_to_real_map[empty_node.ord] = 0
                    else:
                        # Success: Use the syntactic parent's index
                        empty_to_real_map[empty_node.ord] = ord_to_idx.get(anchor_node.ord, 0)

            sentence_entities = []
            processed_mentions = set()

            # 2. Extract mentions using the syntactic map
            for node in tree.descendants_and_empty:
                for mention in node.coref_mentions:
                    if mention in processed_mentions:
                        continue

                    if "," in mention.span:
                        reduce_discontinuous_mention(mention)

                    # Determine indices
                    all_indices = []
                    for w in mention.words:
                        if not w.is_empty():
                            all_indices.append(ord_to_idx[w.ord])
                        else:
                            # Use the syntactic head mapping for empty components
                            all_indices.append(empty_to_real_map.get(w.ord, 0))

                    # Head index logic
                    if mention.head.is_empty():
                        head_idx = empty_to_real_map.get(mention.head.ord, 0)
                        head_type = "zero_node"
                    else:
                        head_idx = ord_to_idx.get(mention.head.ord, 0)
                        head_type = "entity"

                    if not all_indices:
                        all_indices = [head_idx]

                    # CLEAN TEXT: Ignore empty nodes in the string
                    clean_mention_text = " ".join([n.form for n in mention.words if not n.is_empty()])

                    sentence_entities.append({
                        "COREF": mention.entity.eid,
                        "start_token": min(all_indices),
                        "end_token": max(all_indices),
                        "head_token": head_idx,
                        "cat": head_type,
                        "mention_text": clean_mention_text if clean_mention_text else ""
                    })
                    processed_mentions.add(mention)

            sentences_in_doc.append({
                "tokens": sentence_tokens,
                "entities": sentence_entities,
                # "sent_id": tree.sent_id
            })
        dataset_docs.append(sentences_in_doc)

    return dataset_docs

In [3]:
from pathlib import Path
import os

In [4]:
data_directory = "/data-lachesis/common/shared_tasks/crac26/crac26_data"
# for split in ["llm-gold-train", "llm-gold-minidev", "llm-input_blind-minidev"]:
for split in ["llm-input_blind-minitest"]:
    split_directory = Path(data_directory, split)
    ext = ".conllu"  # change if needed
    conllu_datasets = sorted([p.stem for p in Path(split_directory).iterdir()
             if p.is_file() and p.suffix == ext], reverse=False)

    print(f"{ext} files: {len(conllu_datasets)}")

    for dataset_name in tqdm(conllu_datasets):
        DATASET_conllu_file_path = os.path.join(split_directory, dataset_name + ".conllu")
        DATASET_docs = read_data(DATASET_conllu_file_path)
        DATASET_tokens_entities_dataset = extract_dataset(DATASET_docs)

        for DOC_tokens_entities_dataset in DATASET_tokens_entities_dataset:
            coref_mapping = {}
            for SENT_tokens_entities_dataset in DOC_tokens_entities_dataset:
                for entity in SENT_tokens_entities_dataset["entities"]:
                    old_COREF_id = entity["COREF"]
                    if old_COREF_id not in coref_mapping:
                        coref_mapping[old_COREF_id] = len(coref_mapping)
                    entity["COREF"] = coref_mapping[old_COREF_id]

        DATASET_tokens_entities_dataset_path = os.path.join(split_directory, dataset_name + ".tokens_entities_dataset_json")
        with open(DATASET_tokens_entities_dataset_path, "w", encoding="utf-8") as f:
            # Adding indent=2 makes it human-readable; remove it for a smaller file size
            json.dump(DATASET_tokens_entities_dataset, f, indent=2, ensure_ascii=False)

        print(f"    Saved to: {DATASET_tokens_entities_dataset_path}")

.conllu files: 27


  0%|          | 0/27 [00:00<?, ?it/s]03/04/2026 14:29:24 - INFO - root - Reading /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/ca_ancora-corefud-minitest.conllu

  4%|▎         | 1/27 [00:03<01:19,  3.07s/it]03/04/2026 14:29:28 - INFO - root - Reading /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/cs_pcedt-corefud-minitest.conllu


    Saved to: /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/ca_ancora-corefud-minitest.tokens_entities_dataset_json



  7%|▋         | 2/27 [00:06<01:19,  3.19s/it]03/04/2026 14:29:31 - INFO - root - Reading /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/cs_pdt-corefud-minitest.conllu


    Saved to: /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/cs_pcedt-corefud-minitest.tokens_entities_dataset_json


...

    Saved to: /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/pl_pcc-corefud-minitest.tokens_entities_dataset_json



 96%|█████████▋| 26/27 [00:48<00:01,  1.68s/it]03/04/2026 14:30:13 - INFO - root - Reading /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/tr_itcc-corefud-minitest.conllu


    Saved to: /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/ru_rucor-corefud-minitest.tokens_entities_dataset_json



100%|██████████| 27/27 [00:48<00:00,  1.80s/it]

    Saved to: /data-lachesis/common/shared_tasks/crac26/crac26_data/llm-input_blind-minitest/tr_itcc-corefud-minitest.tokens_entities_dataset_json
